# PocketVoice — Live-talk Miku (w-okada)

**Realtime** WebSocket voice changer. Говоришь в мик → слышишь Miku **в реальном времени**.

**Что делать:**
1. Runtime → Change runtime type → **T4 GPU** → Save (один раз)
2. **Run All** (Ctrl+F9)
3. Жди 10-15 мин setup
4. В конце появится `https://*.trycloudflare.com` — открой эту ссылку в Chrome на телефоне
5. В w-okada UI: Model Slot 0 уже стоит Miku — нажми **Start**
6. Разреши доступ к микрофону
7. Говори — слышишь Miku live

Без ngrok токенов. Без аккаунтов. Только Google для Colab.

In [ ]:
# === 1/4. Клон w-okada + системные зависимости ===
%cd /content/
!pip install -q colorama
from colorama import Fore, Style

print(f'{Fore.CYAN}> git clone w-okada/voice-changer…{Style.RESET_ALL}')
!git clone -q https://github.com/w-okada/voice-changer.git
%cd voice-changer/server/

print(f'{Fore.CYAN}> apt: libportaudio2…{Style.RESET_ALL}')
!apt-get -y install libportaudio2 -qq 2>&1 | tail -3
print(f'{Fore.GREEN}> system deps ready{Style.RESET_ALL}')

In [ ]:
# === 2/4. Python deps (~5-7 мин) ===
from colorama import Fore, Style
print(f'{Fore.CYAN}> pre-deps (faiss-gpu, fairseq, pyworld)…{Style.RESET_ALL}')
!pip install -q faiss-gpu fairseq pyworld --no-build-isolation
print(f'{Fore.CYAN}> requirements.txt…{Style.RESET_ALL}')
!pip install -q -r requirements.txt
print(f'{Fore.GREEN}> all python deps installed{Style.RESET_ALL}')

In [ ]:
# === 3/4. Качаем Miku модель в слот 0 ===
import os, urllib.request, json
from colorama import Fore, Style

MIKU_PTH = 'https://huggingface.co/NoCrypt/miku_RVC/resolve/main/1a_miku_default_rvc_(aple)/miku_default_rvc.pth'
MIKU_IDX = 'https://huggingface.co/NoCrypt/miku_RVC/resolve/main/1a_miku_default_rvc_(aple)/added_IVF4457_Flat_nprobe_1_miku_default_rvc_v2.index'

slot_dir = '/content/voice-changer/server/model_dir/RVC/0'
os.makedirs(slot_dir, exist_ok=True)

print(f'{Fore.CYAN}> downloading Miku checkpoint (~52 MB)…{Style.RESET_ALL}')
miku_pth = f'{slot_dir}/miku.pth'
miku_idx = f'{slot_dir}/miku.index'
if not os.path.exists(miku_pth):
    urllib.request.urlretrieve(MIKU_PTH, miku_pth)
if not os.path.exists(miku_idx):
    urllib.request.urlretrieve(MIKU_IDX, miku_idx)
print(f'  miku.pth = {os.path.getsize(miku_pth)//1024//1024} MB')
print(f'  miku.index = {os.path.getsize(miku_idx)//1024//1024} MB')

# Регистрируем slot 0 как Miku, чтобы UI показал её сразу
params = {
    'slotIndex': 0,
    'voiceChangerType': 'RVC',
    'name': 'Miku',
    'description': 'Hatsune Miku (RVC v2, NoCrypt)',
    'modelFile': 'miku.pth',
    'indexFile': 'miku.index',
    'defaultTune': 12,
    'defaultIndexRatio': 0.75,
    'defaultProtect': 0.33,
    'sampleRate': 40000,
    'modelType': 'pyTorchRVCv2',
    'embChannels': 768,
    'embOutputLayer': 12,
    'useFinalProj': False,
    'f0': True,
}
with open(f'{slot_dir}/params.json', 'w') as f:
    json.dump(params, f, indent=2)
print(f'{Fore.GREEN}> Miku slot 0 готов{Style.RESET_ALL}')

# cloudflared для туннеля без ngrok токена
if not os.path.exists('/content/cloudflared'):
    print(f'{Fore.CYAN}> cloudflared…{Style.RESET_ALL}')
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '/content/cloudflared')
    os.chmod('/content/cloudflared', 0o755)
    print(f'{Fore.GREEN}> cloudflared ready{Style.RESET_ALL}')

In [ ]:
# === 4/4. Запуск w-okada сервера + cloudflared туннель ===
import subprocess, time, re, threading, sys, os
from colorama import Fore, Style

os.chdir('/content/voice-changer/server')

print(f'{Fore.CYAN}> Starting w-okada server on :18888…{Style.RESET_ALL}')
server = subprocess.Popen(
    [sys.executable, 'MMVCServerSIO.py',
     '-p', '18888',
     '--https', 'False',
     '--content_vec_500', 'pretrain/checkpoint_best_legacy_500.pt',
     '--hubert_base', 'hubert_base.pt',
     '--rmvpe', 'rmvpe.pt',
     '--colab', 'True'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

ready = False
for line in iter(server.stdout.readline, ''):
    print(line, end='')
    if 'Uvicorn running' in line or '18888' in line or 'Running on' in line:
        ready = True
        break
    if server.poll() is not None:
        print(f'{Fore.RED}[!] server exited{Style.RESET_ALL}')
        break

if ready:
    time.sleep(3)
    print(f'\n{Fore.CYAN}> cloudflared tunnel…{Style.RESET_ALL}')
    tunnel = subprocess.Popen(
        ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:18888', '--no-autoupdate'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    url_re = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
    public_url = None
    for line in iter(tunnel.stdout.readline, ''):
        print(line, end='')
        m = url_re.search(line)
        if m: public_url = m.group(0); break
    if public_url:
        print('\n' + '═'*70)
        print(f'║  🎤 LIVE-TALK MIKU URL: {public_url}')
        print('═'*70)
        print(f'\n👉 Открой ссылку в Chrome на Poco')
        print(f'👉 Model Slot 0 = Miku → Start → разреши mic → говори')
        print(f'👉 Latency 100-300мс на T4 GPU\n')
    def tail(p, tag):
        for ln in iter(p.stdout.readline, ''): print(f'[{tag}] {ln}', end='')
    threading.Thread(target=tail, args=(server,'srv'), daemon=True).start()
    threading.Thread(target=tail, args=(tunnel,'tun'), daemon=True).start()
    print(f'{Fore.GREEN}[ready] Держу. Не закрывай вкладку.{Style.RESET_ALL}')
    try:
        while True:
            time.sleep(60)
            if server.poll() is not None: print('[!] server died'); break
            if tunnel.poll() is not None: print('[!] tunnel died'); break
    except KeyboardInterrupt:
        server.terminate(); tunnel.terminate()